In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env file in this folder

# Or set it directly here for quick demos:
# os.environ["GROQ_API_KEY"] = "your-key-here"

assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in a .env file or above"
print("API key loaded.")

API key loaded.


In [2]:
from pypdf import PdfReader

# reader = PdfReader("NIPS-2017-attention-is-all-you-need-Paper.pdf")
reader = PdfReader("telecom_guide.pdf")

pages = []
for page in reader.pages:
    pages.append(page.extract_text())

print(f"Loaded {len(pages)} pages")
print(pages[0][:500])

Loaded 9 pages
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,       # 👈 set here
    chunk_overlap=30,     # 👈 set here
    separators=["\n\n", "\n", ".", " "],
)
clean_pages = [
    page.replace("Telecom Technical Reference Guide  - Internal Use Only", "")
    for page in pages
]
docs = [Document(page_content=page) for page in clean_pages]
chunks = splitter.split_documents(docs)

print(f"Total chunks: {len(chunks)}  (from {len(pages)} pages)")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print("\n--- Example chunk ---")
print(chunks[5].page_content)

Total chunks: 36  (from 9 pages)
Avg chunk length: 490 chars

--- Example chunk ---
Step 2  - Rule out a temporary network glitch. Toggle airplane mode on for 10 seconds then off. This forces the
device to re-register on the network, which resolves many transient issues.
Step 3  - Check APN settings. The Access Point Name (APN) tells the device how to connect to the carrier's data
network. Incorrect APN settings are a common cause of data connectivity failure after a phone reset or SIM swap.
Navigate to Settings > Mobile Network > APN and verify the APN name, username, and password match the
carrier's published settings.


In [4]:
chunks[0].page_content

'Telecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [5]:
chunks[1].page_content

'1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app\ndownloads at speeds of 1-14 Mbps. This generation established the foundation for smartphone adoption\nworldwide.'

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model (downloads ~90 MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in Chroma (in memory)...")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

Loading embedding model (downloads ~90 MB on first run)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding all chunks and storing in Chroma (in memory)...
Vector store ready. 36 vectors stored.


In [7]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10  # fetch more, then filter diversity
    }
)

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network us

--- Chunk 2 ---
7. Billing System Architecture and Common Disputes
Understanding how the billing system works helps agents identify the root cause of billing disputes quickly and
accurately.
Event-Based Billing: Every billable event  - a phone call, an SMS, a data session  - generates a Call Detail Record
(CDR) at 

--- Chunk 3 ---
protocols. 5G's use of HTTPS-based APIs (Service Based Architecture) substantially reduces this attack surface.
SIM Swap Fraud: Described in Section 5. Key mitigation: enforce strict in-person or multi-factor remote identity
verification before any SIM replacement. Fl

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)

# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [9]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: International roaming allows your device to connect to partner networks in visited countries when outside your home network's coverage. Here's how it works and what charges to expect:

**How It Works:**  
1. **Authentication:** The visited network verifies your subscription via signaling protocols (SS7/Diameter), and your home network authorizes service.  
2. **Service Usage:** Data, voice, and SMS are billed based on your plan and roaming agreements.  

**Charges:**  
- **Data/Voice/SMS:** Specific rates apply, though exact pricing depends on your plan.  
- **Bundle Pricing:** If you purchase a roaming bundle, using some data still counts toward the bundle price, but pre-bundle usage is charged at standard rates (not reversed).  
- **Bill Shock Alerts:** Automated alerts notify you when roaming charges exceed $20, $50, or $100 thresholds.  

**Key Notes:**  
- Ensure roaming is enabled on your account and dev